# Whisper ASR + LLM — Colab to local server

## 1. Install dependencies

In [ ]:
!python -m pip install bitsandbytes --prefer-binary --extra-index-url=https://jllllll.github.io/bitsandbytes-windows-webui

!pip install --upgrade pip
!pip install transformers accelerate
!pip install peft==0.10.0 trl==0.8.6
!pip install -U gradio
!pip install evaluate
!pip install soundfile
!pip install librosa

!pip install git+https://github.com/openai/whisper.git -q

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Paths & config

In [ ]:
import os, sys

DRIVE_ROOT       = '/content/drive/MyDrive'
BASE_DIR         = f'{DRIVE_ROOT}/capstone_design'
SRC_DIR          = f'{BASE_DIR}/pipeline_src'
HF_REPO_WHISPER  = 'minsu0567/Capstone-Design-Whisper-Dialect'
HF_REPO_LLM      = 'minsu0567/Capstone-Design-Llama3.2-Command'
LOCAL_SERVER_URL = 'https://8db3-221-161-190-89.ngrok-free.app/test'

assert os.path.isfile(f'{SRC_DIR}/voice_command_pipeline.py'), \
    f'Missing: {SRC_DIR}/voice_command_pipeline.py'

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print('Paths OK.')
print('server  :', LOCAL_SERVER_URL)

## 4. Load Whisper (merged)

In [ ]:
import torch
from transformers import (AutoTokenizer, WhisperForConditionalGeneration,
                          WhisperProcessor)

model = WhisperForConditionalGeneration.from_pretrained(
    HF_REPO_WHISPER,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()

processor = WhisperProcessor.from_pretrained(HF_REPO_WHISPER)
whisper_tokenizer = AutoTokenizer.from_pretrained(HF_REPO_WHISPER)

print('Whisper ready on', next(model.parameters()).device)

## 5. Load LLM (merged)

In [ ]:
from transformers import AutoModelForCausalLM

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

llm_model = AutoModelForCausalLM.from_pretrained(
    HF_REPO_LLM,
    torch_dtype=DTYPE,
    device_map='auto',
)
llm_model.eval()

tokenizer = AutoTokenizer.from_pretrained(HF_REPO_LLM)
llm_model.config.pad_token_id = tokenizer.pad_token_id

print('LLM ready on', next(llm_model.parameters()).device, '|', DTYPE)

## 6. Launch Gradio

In [ ]:
from voice_command_pipeline import build_interface

interface = build_interface(
    whisper_model     = model,
    processor         = processor,
    whisper_tokenizer = whisper_tokenizer,
    llm_model         = llm_model,
    llm_tokenizer     = tokenizer,
    server_url        = LOCAL_SERVER_URL,
)
interface.launch(share=True)